# Sesión 2 · De 12 000 filas a una distribución
**Análisis Estadístico de la Información — Especialización en Big Data (UCompensar)**

Este cuaderno (*notebook*) acompaña las [diapositivas de la sesión](https://jotamao1985.github.io/ANaLISIS-ESTADiSTICO-DE-LA-INFORMACIoN_Ucomp/diapositivas/sesion-2-de-las-filas-a-la-distribucion.html).
Cada celda lleva entre corchetes el mismo identificador que el código proyectado, así que puedes seguir
la clase celda a celda. La portada del curso está en https://jotamao1985.github.io/ANaLISIS-ESTADiSTICO-DE-LA-INFORMACIoN_Ucomp/

1. **Archivo → Guardar una copia en Drive** para que tus cambios no se pierdan.
2. Ejecuta las celdas **en orden** con **Mayús + Enter**. Varias usan resultados de las anteriores.
   Si Colab avisa que el cuaderno no es de Google, elige **Ejecutar de todos modos**.
3. Si Colab se reinicia o una celda falla al repetirla, usa **Entorno de ejecución → Ejecutar anteriores**.
4. La primera celda crea los datos del caso *Tingua Domicilios S.A.S.*, una empresa **ficticia**
   (sus datos se inventan con una semilla fija y no corresponden a personas ni a empresas reales),
   y les da a las gráficas los colores del curso.

In [ ]:
#@title ▶ Crea los datos del caso (ejecútala sin modificar) { display-mode: "form" }
"""Datos del caso «Tingua Domicilios S.A.S.» (empresa FICTICIA).

Genera datos/pedidos.csv: los 12 000 pedidos que la plataforma entregó entre el
1 de enero y el 30 de junio de 2026 en cinco ciudades. Todo se inventa con una
semilla fija: los datos no corresponden a personas ni a empresas reales.

Solo usa la biblioteca estándar y, del generador aleatorio, solo random(), que
es lo único que Python garantiza igual entre versiones. Por eso la tabla sale
idéntica en Colab y en cualquier computador.

El caso sirve para todo el curso: variables de todos los tipos (sesión 1),
asimetrías y algunos atípicos (medidas y datos atípicos), una población
completa de la que se pueden sacar muestras (muestreo) y un tiempo de entrega
que depende de la distancia, la lluvia y otras variables (regresión).
"""

import csv
import math
import random
from datetime import date, timedelta
from pathlib import Path

SEMILLA = 2026
N_PEDIDOS = 12_000
INICIO = date(2026, 1, 1)
DIAS = 181  # del 1 de enero al 30 de junio de 2026

COLUMNAS = ["id_pedido", "fecha", "franja", "ciudad", "categoria", "n_productos",
            "valor_pedido", "medio_pago", "estrato", "suscriptor", "distancia_km",
            "lluvia", "temperatura_c", "tiempo_entrega_min", "calificacion"]

# ciudad: (peso, temperatura media °C, desviación, prob. de lluvia, minutos extra, factor de distancia)
CIUDADES = {
    "Bogotá":       (0.40, 14.5, 2.2, 0.30, 5.0, 1.15),
    "Medellín":     (0.22, 22.5, 2.3, 0.28, 2.0, 1.00),
    "Cali":         (0.16, 25.0, 2.3, 0.22, 0.0, 1.00),
    "Barranquilla": (0.12, 29.0, 1.8, 0.12, -1.0, 0.95),
    "Bucaramanga":  (0.10, 23.5, 2.2, 0.20, -2.0, 0.90),
}
LLUVIA_MES = {1: 0.6, 2: 0.7, 3: 1.0, 4: 1.4, 5: 1.3, 6: 0.8}

# franja: (peso, ajuste de temperatura °C, minutos extra)
FRANJAS = {
    "Mañana":   (0.15, -1.5, 0.0),
    "Almuerzo": (0.35, 1.5, 4.0),
    "Tarde":    (0.20, 1.0, 0.0),
    "Noche":    (0.30, -2.0, 3.0),
}

# categoría: (productos extra en promedio, precio medio por producto en COP, minutos de preparación)
CATEGORIAS = {
    "Restaurante": (1.3, 27_000, 3.0),
    "Mercado":     (6.0, 8_500, 4.0),
    "Farmacia":    (0.9, 21_000, 0.0),
    "Mascotas":    (1.2, 36_000, 1.0),
    "Floristería": (0.2, 88_000, 2.0),
}
# Qué se pide en cada franja (mismo orden que CATEGORIAS)
CATEGORIA_POR_FRANJA = {
    "Mañana":   (0.30, 0.35, 0.20, 0.10, 0.05),
    "Almuerzo": (0.78, 0.08, 0.07, 0.04, 0.03),
    "Tarde":    (0.40, 0.25, 0.15, 0.12, 0.08),
    "Noche":    (0.70, 0.10, 0.12, 0.05, 0.03),
}
ESTRATOS = {1: 0.06, 2: 0.22, 3: 0.33, 4: 0.20, 5: 0.11, 6: 0.08}


class _Azar:
    """Lo que hace falta del azar, construido solo sobre random()."""

    def __init__(self, semilla):
        self.r = random.Random(semilla)

    def u(self):
        return self.r.random()

    def elegir(self, opciones, pesos):
        x, acum = self.u() * sum(pesos), 0.0
        for opcion, peso in zip(opciones, pesos):
            acum += peso
            if x < acum:
                return opcion
        return opciones[-1]

    def normal(self, media=0.0, desv=1.0):
        # Box-Muller
        u1, u2 = 1.0 - self.u(), self.u()
        return media + desv * math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)

    def exponencial(self, media):
        return -media * math.log(1.0 - self.u())

    def poisson(self, lam):
        # Knuth: suficiente para medias pequeñas
        limite, k, p = math.exp(-lam), 0, 1.0
        while True:
            p *= self.u()
            if p <= limite:
                return k
            k += 1


def _redondear(x, decimales=0):
    factor = 10 ** decimales
    return math.floor(x * factor + 0.5) / factor


def _pedido(az, dias, pesos_dias):
    fecha = az.elegir(dias, pesos_dias)
    ciudad = az.elegir(list(CIUDADES), [v[0] for v in CIUDADES.values()])
    _, t_media, t_desv, p_lluvia, min_ciudad, f_dist = CIUDADES[ciudad]
    franja = az.elegir(list(FRANJAS), [v[0] for v in FRANJAS.values()])
    _, t_franja, min_franja = FRANJAS[franja]
    categoria = az.elegir(list(CATEGORIAS), CATEGORIA_POR_FRANJA[franja])
    extra, precio, min_prep = CATEGORIAS[categoria]
    estrato = az.elegir(list(ESTRATOS), list(ESTRATOS.values()))

    suscriptor = az.u() < 0.10 + 0.06 * (estrato - 1)
    p_efectivo = 0.45 - 0.07 * (estrato - 1)
    medio_pago = az.elegir(["Efectivo", "Billetera digital", "Tarjeta"],
                           [p_efectivo, 0.25, 1 - p_efectivo - 0.25])

    n = 1 + az.poisson(extra)
    if categoria in ("Mercado", "Restaurante") and az.u() < 0.003:
        n = 20 + int(az.u() * 26)  # pedido corporativo: pocos, pero existen
    valor = n * precio * math.exp(az.normal(0, 0.22)) * (1 + 0.07 * (estrato - 3))
    valor = max(3_000, _redondear(valor / 100) * 100)

    distancia = math.exp(az.normal(math.log(3.0), 0.5)) * f_dist
    distancia = min(14.0, max(0.4, _redondear(distancia, 1)))

    lluvia = az.u() < p_lluvia * LLUVIA_MES[fecha.month]
    temperatura = _redondear(az.normal(t_media + t_franja - 1.8 * lluvia, t_desv), 1)

    tiempo = (11 + 3.1 * distancia + 0.45 * min(n, 15) + min_prep + min_franja + min_ciudad
              + 7.5 * lluvia - 2.5 * suscriptor + az.normal(0, 4) + az.exponencial(3))
    if az.u() < 0.005:
        tiempo += 55 + 95 * az.u()  # pedidos que se demoraron muchísimo
    tiempo = max(6.0, _redondear(tiempo, 1))

    latente = 6.4 - 0.065 * tiempo + 0.2 * suscriptor + az.normal(0, 0.6)
    estrellas = int(min(5, max(1, math.floor(latente + 0.5))))
    califico = az.u() < (0.80 if tiempo > 55 else 0.58)

    return {
        "fecha": fecha, "franja": franja, "ciudad": ciudad, "categoria": categoria,
        "n_productos": n, "valor_pedido": int(valor), "medio_pago": medio_pago,
        "estrato": estrato, "suscriptor": "Sí" if suscriptor else "No",
        "distancia_km": distancia, "lluvia": "Sí" if lluvia else "No",
        "temperatura_c": temperatura, "tiempo_entrega_min": tiempo,
        "calificacion": estrellas if califico else "",
    }


def crear_datos(carpeta="datos"):
    """Escribe <carpeta>/pedidos.csv y devuelve su ruta."""
    az = _Azar(SEMILLA)
    dias = [INICIO + timedelta(days=d) for d in range(DIAS)]
    pesos_dias = [1.35 if d.weekday() >= 4 else 1.0 for d in dias]  # viernes a domingo, más pedidos
    pedidos = [_pedido(az, dias, pesos_dias) for _ in range(N_PEDIDOS)]
    orden_franja = list(FRANJAS)
    pedidos.sort(key=lambda p: (p["fecha"], orden_franja.index(p["franja"])))

    ruta = Path(carpeta) / "pedidos.csv"
    ruta.parent.mkdir(parents=True, exist_ok=True)
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        escritor = csv.DictWriter(f, fieldnames=COLUMNAS)
        escritor.writeheader()
        for i, p in enumerate(pedidos, start=1):
            escritor.writerow({"id_pedido": f"P{i:05d}", **p, "fecha": p["fecha"].isoformat()})
    miles = f"{N_PEDIDOS:,}".replace(",", "\u202f")  # 12 000, con espacio fino
    print(f"Listo: {ruta} con {miles} pedidos y {len(COLUMNAS)} columnas")
    return ruta


"""Ayudas comunes a todos los notebooks del curso (van en la celda oculta, tras genera_datos.py).

Las de cada sesión están en ayudas_sesionN.py.
"""


def _aviso_version():
    """Las diapositivas se ejecutaron con pandas 2.2; con pandas 3 algunas salidas cambian."""
    import pandas as pd

    if int(pd.__version__.split(".")[0]) >= 3:
        print(f"Aviso: este Colab trae pandas {pd.__version__} y las diapositivas usan pandas 2.2. "
              "Algunas salidas se verán un poco distintas (por ejemplo, el texto sale como str "
              "y no como object).")


_aviso_version()


"""Sesión 2: estilo de las gráficas del curso y verificadores de la práctica.

Van dentro de la celda oculta del notebook, junto con genera_datos.py y ayudas.py. Cada
verificador recibe lo que el estudiante calculó, dice qué está bien y, en lo demás, da una
pista sin la respuesta. Nunca se cae: un «...» sin reemplazar o un tipo inesperado se avisa.
"""


def _estilo_graficas():
    """Colores del curso y letras que se leen en Colab y proyectadas."""
    import matplotlib as mpl
    import matplotlib.pyplot as plt

    plt.rcParams.update({
        "axes.prop_cycle": mpl.cycler(color=["#8A2E9E", "#F46201", "#566B7F"]),
        "figure.figsize": (6.4, 3.6),
        "figure.constrained_layout.use": True,
        "font.size": 12,
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#94A3B8",
        "axes.labelcolor": "#334155",
        "xtick.color": "#475569",
        "ytick.color": "#475569",
        "text.color": "#1E293B",
    })


_estilo_graficas()


def _mostrar(tabla):
    try:
        from IPython.display import display
        display(tabla)
    except ImportError:
        print(tabla.to_string())


def _serie(x):
    """La serie que entregó el estudiante, o None si no es una."""
    import pandas as pd

    if isinstance(x, pd.DataFrame) and x.shape[1] == 1:
        x = x.iloc[:, 0]
    if not isinstance(x, pd.Series):
        return None
    if isinstance(x.index, pd.MultiIndex) and x.index.nlevels == 1:  # pedidos[["col"]].value_counts()
        x = x.set_axis(x.index.get_level_values(0))
    return x


def _numerica(serie):
    import pandas as pd

    return pd.api.types.is_numeric_dtype(serie) and not pd.api.types.is_bool_dtype(serie)


def _valores(serie, indice):
    """Los valores de la serie en el orden de `indice` (claves comparadas como texto)."""
    texto = lambda c: str(c).removesuffix(".0")
    por_clave = {texto(k): v for k, v in serie.items()}
    try:
        return [float(por_clave[texto(k)]) for k in indice]
    except (KeyError, TypeError, ValueError):
        return None


def _cerca(a, b, tolerancia):
    return a is not None and len(a) == len(b) and all(abs(x - y) <= tolerancia for x, y in zip(a, b))


def revisar_nominal(conteo=..., porcentaje=...):
    """Ejercicio 1: frecuencia absoluta y relativa (en %) de medio_pago."""
    import pandas as pd

    original = pd.read_csv("datos/pedidos.csv")
    esperado = original["medio_pago"].value_counts()
    n = len(original)
    absolutas = [float(v) for v in esperado]
    relativas = [100 * v / n for v in absolutas]
    bien = 0

    c = _serie(conteo)
    if conteo is Ellipsis:
        print("·  conteo: todavía tiene ...")
    elif c is None:
        print("✗  conteo: tiene que ser una serie de pandas, como la que devuelve value_counts()")
    elif len(c) == n:
        print("✗  conteo: esa es la columna entera, un valor por pedido: falta contarla")
    elif _valores(c, esperado.index) is None or len(c) != len(esperado):
        print("✗  conteo: sus categorías no son las de medio_pago: ¿contaste la columna correcta?")
    elif not _numerica(c):
        print("✗  conteo: sus valores son texto; déjalos como números (sin astype(str))")
    elif _cerca(_valores(c, esperado.index), absolutas, 0):
        print("✓  conteo")
        bien += 1
    elif abs(c.sum() - 1) < 1e-6 or abs(c.sum() - 100) < 0.5:
        print("✗  conteo: eso ya es una frecuencia relativa; la absoluta cuenta pedidos")
    else:
        print("✗  conteo: los números no cuadran: ¿contaste todos los pedidos?")

    p = _serie(porcentaje)
    vp = None if p is None else _valores(p, esperado.index)
    if porcentaje is Ellipsis:
        print("·  porcentaje: todavía tiene ...")
    elif p is None:
        print("✗  porcentaje: tiene que ser una serie de pandas, como la que devuelve value_counts()")
    elif len(p) == n:
        print("✗  porcentaje: esa es la columna entera, un valor por pedido: falta contarla")
    elif vp is None or len(p) != len(esperado):
        print("✗  porcentaje: sus categorías no son las de medio_pago")
    elif not _numerica(p):
        print("✗  porcentaje: sus valores son texto; déjalos como números (sin astype(str) ni « %»)")
    elif _cerca(vp, relativas, 0.051):
        print("✓  porcentaje")
        bien += 1
    elif _cerca(vp, [r / 100 for r in relativas], 0.00051):
        print("✗  porcentaje: está como proporción (suma 1); el ejercicio la pide en %")
    elif _cerca(vp, absolutas, 0):
        print("✗  porcentaje: eso es la frecuencia absoluta; ¿qué hace normalize=True?")
    elif _cerca(vp, [a * 100 for a in absolutas], 0):
        print("✗  porcentaje: multiplicaste por 100 los conteos; antes hay que dividir entre el total")
    elif _cerca(vp, relativas, 0.51):
        print("✗  porcentaje: casi: perdiste precisión al redondear; deja al menos un decimal en el %")
    else:
        print("✗  porcentaje: no cuadra: ¿sobre qué total dividiste?")

    if bien == 2:
        tabla = pd.DataFrame({"pedidos": [int(v) for v in absolutas], "%": vp},
                             index=esperado.index)
        _mostrar(tabla.round({"%": 1}))
        print("✓  Tabla lista. ¿Cuál es la moda?")


def revisar_acumulada(conteo=..., acumulado=...):
    """Ejercicio 2: frecuencias del estrato en su orden, con el % acumulado."""
    import pandas as pd

    original = pd.read_csv("datos/pedidos.csv")
    orden = [1, 2, 3, 4, 5, 6]
    absolutas = [float((original["estrato"] == e).sum()) for e in orden]
    n = len(original)
    acumuladas = [100 * sum(absolutas[: i + 1]) / n for i in range(6)]
    bien = 0

    c = _serie(conteo)
    if conteo is Ellipsis:
        print("·  conteo: todavía tiene ...")
    elif c is None:
        print("✗  conteo: tiene que ser una serie de pandas, como la que devuelve value_counts()")
    elif len(c) == n:
        print("✗  conteo: esa es la columna entera, un valor por pedido: falta contarla")
    elif _valores(c, orden) is None or len(c) != 6:
        print("✗  conteo: sus categorías no son los estratos 1 a 6: ¿contaste la columna correcta?")
    elif not _numerica(c):
        print("✗  conteo: sus valores son texto; déjalos como números (sin astype(str))")
    elif not _cerca(_valores(c, orden), absolutas, 0):
        print("✗  conteo: los números no cuadran; la frecuencia absoluta cuenta pedidos")
    elif [str(k).removesuffix(".0") for k in c.index] != [str(e) for e in orden]:
        if list(c) == sorted(c, reverse=True):
            print("✗  conteo: los valores están bien, pero van por tamaño: una ordinal va en su "
                  "orden (1 a 6)")
        else:
            print("✗  conteo: los valores están bien, pero no van del 1 al 6. Si recargaste "
                  "pedidos, vuelve a ejecutar [cargar]: ahí estrato queda como categoría ordenada")
    else:
        print("✓  conteo, en su orden")
        bien += 1

    a = _serie(acumulado)
    valores = None
    if a is not None and _numerica(a):
        valores = [float(v) for v in a]
    if acumulado is Ellipsis:
        print("·  acumulado: todavía tiene ...")
    elif a is None:
        print("✗  acumulado: tiene que ser una serie de pandas (la que devuelve cumsum())")
    elif not _numerica(a):
        print("✗  acumulado: sus valores son texto; déjalos como números (sin astype(str) ni « %»)")
    elif len(a) != 6:
        print("✗  acumulado: debe tener un valor por estrato, seis en total")
    elif _cerca(valores, acumuladas, 0.151):
        print("✓  acumulado")
        bien += 1
    elif _cerca(valores, [x / 100 for x in acumuladas], 0.0016):
        print("✗  acumulado: está como proporción (termina en 1); el ejercicio lo pide en %")
    elif abs(valores[-1] - n) < 0.5:
        print("✗  acumulado: acumulaste pedidos; el ejercicio pide el % acumulado")
    elif valores[-1] > 0 and _cerca([100 * v / valores[-1] for v in valores], acumuladas, 0.151):
        print("✗  acumulado: los pasos cuadran, pero no termina en 100 %: ¿sobre qué total "
              "dividiste? (si usaste n, revisa cuánto vale)")
    elif _cerca(valores, acumuladas, 0.6):
        print("✗  acumulado: casi: perdiste precisión al redondear; redondea solo al final")
    elif abs(valores[-1] - 100) < 0.2:
        print("✗  acumulado: termina en 100, pero los pasos no cuadran: ¿acumulaste en el "
              "orden del estrato?")
    else:
        print("✗  acumulado: el último valor tiene que ser 100 %: acumula los %, no otra cosa")

    if bien == 2:
        tabla = pd.DataFrame({"pedidos": [int(v) for v in absolutas], "% acumulado": valores},
                             index=pd.Index(orden, name="estrato"))
        _mostrar(tabla.round({"% acumulado": 1}))
        print("✓  Tabla lista. Responde abajo: ¿qué % de los pedidos va a direcciones de estrato 3 "
              "o menos?")


def revisar_intervalos(cortes=...):
    """Ejercicio 3: intervalos de 2 km, cerrados a la izquierda, que cubran todos los pedidos."""
    import math
    import pandas as pd

    if cortes is Ellipsis:
        print("·  cortes: todavía tiene ...")
        return
    try:
        cortes = [float(x) for x in cortes]
    except (TypeError, ValueError):
        print("✗  cortes: escribe una lista de números, como [0, 2, 4] o list(range(0, 10, 2))")
        return
    if len(cortes) < 2:
        print("✗  cortes: hacen falta al menos dos cortes: el inicio y el fin de cada intervalo")
        return
    if any(math.isnan(x) for x in cortes):
        print("✗  cortes: hay un valor que no es un número (nan)")
        return
    if any(math.isinf(x) for x in cortes):
        print("✗  cortes: una clase abierta (inf) sirve cuando la cola no tiene techo; aquí todos los "
              "intervalos miden 2 km: ¿cuál es la distancia máxima?")
        return
    distancia = pd.read_csv("datos/pedidos.csv")["distancia_km"]
    if any(b <= a for a, b in zip(cortes, cortes[1:])):
        print("✗  cortes: tienen que ir de menor a mayor")
        return
    if cortes[0] != 0:
        print("✗  cortes: el primer intervalo empieza en 0 km")
        return
    if any(abs((b - a) - 2) > 1e-9 for a, b in zip(cortes, cortes[1:])):
        print("✗  cortes: cada intervalo mide 2 km: de 2 en 2")
        return
    fuera = int((distancia >= cortes[-1]).sum())
    if fuera:
        print(f"✗  cortes: se quedan por fuera {fuera} pedidos. Con intervalos [a, b), el último "
              "no incluye su borde derecho: ¿cuál es la distancia máxima?")
        return
    clases = pd.cut(distancia, bins=cortes, right=False)
    tabla = clases.value_counts(sort=False).to_frame("pedidos")
    tabla["%"] = (tabla["pedidos"] / len(distancia) * 100).round(1)
    vacias = int((tabla["pedidos"] == 0).sum())
    _mostrar(tabla)
    print(f"✓  Los {len(distancia):,} pedidos caben en {len(tabla)} intervalos de 2 km".replace(",", " "))
    if vacias:
        sobran = "Sobra 1 intervalo vacío" if vacias == 1 else f"Sobran {vacias} intervalos vacíos"
        print(f"·  {sobran} al final: puedes terminar antes")


def revisar_cruzada(mezcla=...):
    """Ejercicio 4: qué se pide en cada franja: franja × categoría, en % por fila."""
    import pandas as pd

    if mezcla is Ellipsis:
        print("·  mezcla: todavía tiene ...")
        return
    if not isinstance(mezcla, pd.DataFrame):
        print("✗  mezcla: tiene que ser una tabla (DataFrame), como la que devuelve pd.crosstab")
        return
    original = pd.read_csv("datos/pedidos.csv")
    cuenta = pd.crosstab(original["franja"], original["categoria"])
    texto = lambda x: {str(v) for v in x}
    if texto(mezcla.index) == texto(cuenta.columns) and texto(mezcla.columns) == texto(cuenta.index):
        print("✗  mezcla: las franjas deben ir en las filas y las categorías en las columnas: "
              "intercambia los dos primeros argumentos")
        return
    if (texto(mezcla.index) != texto(cuenta.index) or texto(mezcla.columns) != texto(cuenta.columns)
            or mezcla.index.duplicated().any() or mezcla.columns.duplicated().any()):
        print("✗  mezcla: las filas tienen que ser las franjas y las columnas, las categorías "
              "(sin totales)")
        return
    tuya = mezcla.copy()
    tuya.index, tuya.columns = tuya.index.astype(str), tuya.columns.astype(str)
    try:
        tuya = tuya.loc[cuenta.index, cuenta.columns].astype(float)
    except (TypeError, ValueError):
        print("✗  mezcla: tiene texto; deja los números (sin astype(str) ni « %»)")
        return
    por_fila = cuenta.div(cuenta.sum(axis=1), axis=0)
    por_columna = cuenta / cuenta.sum()
    del_total = cuenta / cuenta.to_numpy().sum()
    igual = lambda x, y, tol: bool(((x - y).abs() <= tol).all().all())
    if igual(tuya, por_fila, 0.0006) or igual(tuya / 100, por_fila, 0.0006):  # en proporción o en %
        orden = ["Mañana", "Almuerzo", "Tarde", "Noche"]
        _mostrar((por_fila.loc[orden] * 100).round(1))
        print("✓  Cada fila suma 100 %: es la mezcla de categorías de cada franja")
    elif igual(tuya, por_columna, 0.006) or igual(tuya / 100, por_columna, 0.006):
        print("✗  mezcla: cada columna suma 100 %. La pregunta es por la mezcla de cada franja: "
              "¿qué tiene que sumar 100 %?")
    elif igual(tuya, del_total, 0.006) or igual(tuya / 100, del_total, 0.006):
        print("✗  mezcla: toda la tabla suma 100 %. La pregunta es por la mezcla de cada franja: "
              "¿qué tiene que sumar 100 %?")
    elif igual(tuya, cuenta, 0.5):
        print("✗  mezcla: son conteos; falta normalize")
    elif igual(tuya, por_fila, 0.006) or igual(tuya / 100, por_fila, 0.006):
        print("✗  mezcla: casi: perdiste precisión al redondear; deja al menos un decimal en el %")
    else:
        print("✗  mezcla: los valores no cuadran: revisa las columnas que cruzaste")

# Crea datos/pedidos.csv: los mismos 12 000 pedidos de la sesión 1
ruta = crear_datos("datos")

## 1 · Los datos, con sus escalas

Cargamos los pedidos y declaramos las escalas, como en la práctica de la sesión 1: el orden de
`franja`, `estrato` y `calificacion` decide el orden de sus tablas.

In [ ]:
# [cargar]
import pandas as pd
import matplotlib.pyplot as plt

pedidos = pd.read_csv("datos/pedidos.csv")
# Las escalas que declaramos en la sesión 1
franjas = ["Mañana", "Almuerzo", "Tarde", "Noche"]
pedidos["franja"] = pd.Categorical(
    pedidos["franja"], categories=franjas, ordered=True)
pedidos["estrato"] = pd.Categorical(
    pedidos["estrato"], categories=[1, 2, 3, 4, 5, 6], ordered=True)
pedidos["calificacion"] = pd.Categorical(
    pedidos["calificacion"], categories=[1, 2, 3, 4, 5], ordered=True)
pedidos.shape

### Diccionario de datos (el mismo de la sesión 1)

| Columna | Qué registra | Valores o unidad |
|---|---|---|
| `id_pedido` | Código del pedido | P00001, P00002… |
| `fecha` | Día del pedido | del 1 de enero al 30 de junio de 2026 |
| `franja` | Momento del día en que se pidió | Mañana, Almuerzo, Tarde, Noche |
| `ciudad` | Ciudad de entrega | Bogotá, Medellín, Cali, Barranquilla, Bucaramanga |
| `categoria` | Tipo de comercio que despacha | Restaurante, Mercado, Farmacia, Mascotas, Floristería |
| `n_productos` | Unidades pedidas | conteo |
| `valor_pedido` | Valor de los productos, sin domicilio ni propina | pesos (COP) |
| `medio_pago` | Cómo se pagó | Efectivo, Tarjeta, Billetera digital |
| `estrato` | Estrato de la **dirección de entrega** (lo asigna el municipio a la vivienda) | 1 a 6 |
| `suscriptor` | Si quien pide paga la suscripción de Tingua | Sí / No |
| `distancia_km` | Distancia por vía del comercio a la dirección | kilómetros; la app registra hasta 14: una ruta más larga queda como 14 |
| `lluvia` | Si llovía en la zona durante la entrega | Sí / No |
| `temperatura_c` | Temperatura en la zona al momento de la entrega | grados Celsius |
| `tiempo_entrega_min` | Minutos desde que se confirma el pedido hasta que llega | minutos |
| `calificacion` | Estrellas que dio quien pidió; vacía si no calificó | 1 a 5 |

## 2 · Variables cualitativas

### Nominal: cuántos pedidos hay de cada categoría

La **frecuencia absoluta** $n_i$ cuenta los pedidos de cada categoría; la **relativa**,
$f_i = n_i / n$, dice qué parte del total son. Sumadas, dan $n$ y 1 (el 100 %).

In [ ]:
# [nominal]
conteo = pedidos["categoria"].value_counts()
porcentaje = pedidos["categoria"].value_counts(normalize=True) * 100
tabla = pd.DataFrame({"pedidos": conteo, "%": porcentaje.round(1)})
tabla

In [ ]:
# [barras_nominal]
# Una nominal no tiene orden: las barras van de mayor a menor
ax = conteo.sort_values().plot.barh(width=0.6)
ax.bar_label(ax.containers[0], padding=3)
ax.set_title("Pedidos por categoría de comercio")
ax.set_xlabel("Pedidos")
ax.set_ylabel("")
plt.show()

### Ordinal: el orden de la tabla lo pone la escala

En una ordinal las categorías van en su orden, no por tamaño, y se pueden **acumular**: el
**% acumulado** $F_i$ dice qué parte de los pedidos está en esa categoría **o en una anterior**.

In [ ]:
# [ordinal]
# sort=False: en el orden de la escala (1 a 5), no por tamaño
estrellas = pedidos["calificacion"].value_counts(sort=False)
n = estrellas.sum()          # solo los pedidos calificados
tabla = pd.DataFrame({
    "pedidos": estrellas,
    "%": (estrellas / n * 100).round(1),
    "% acumulado": (estrellas.cumsum() / n * 100).round(1),
})
tabla

### ¿Sobre qué total?

`value_counts` deja fuera, sin avisar, los pedidos sin calificación: el 37,4 % de cinco estrellas es
de los pedidos **calificados**. Con `dropna=False`, el total son todos los pedidos.

In [ ]:
# [total]
# value_counts deja fuera los faltantes (NaN); dropna=False los cuenta
todas = pedidos["calificacion"].value_counts(
    normalize=True, sort=False, dropna=False)
(todas * 100).round(1)

In [ ]:
# [barras_ordinal]
# Una ordinal sí tiene orden: las barras van en el orden del día
franja = pedidos["franja"].value_counts(sort=False)
ax = franja.plot.bar(width=0.6, rot=0)
ax.set_title("Pedidos por franja, en el orden del día")
ax.set_xlabel("")
ax.set_ylabel("Pedidos")
plt.show()

## 3 · Variables cuantitativas

### Discreta: una fila por valor, mientras sean pocos

In [ ]:
# [discreta]
productos = pedidos["n_productos"].value_counts().sort_index()
print("Valores distintos:", len(productos),
      "· del", productos.index.min(), "al", productos.index.max())
productos.head(6)

### Continua: se agrupa en intervalos

El tiempo toma cientos de valores distintos: una tabla con una fila por valor no resume nada.
Se agrupa en **intervalos** (clases) que no se traslapan y cubren todos los datos. La regla de
Sturges sugiere cuántos: $k \approx 1 + \log_2 n$.

In [ ]:
# [distintos]
tiempo = pedidos["tiempo_entrega_min"]
print("Valores distintos:", tiempo.nunique(), "de", len(tiempo))
print("Del", tiempo.min(), "al", tiempo.max(), "minutos")

In [ ]:
# [sturges]
import math

k = 1 + math.log2(len(tiempo))
print(f"Sturges sugiere unas {k:.1f} clases")

In [ ]:
# [intervalos]
cortes = [0, 10, 20, 30, 40, 50, 60, 70, 80, float("inf")]
clases = pd.cut(tiempo, bins=cortes, right=False)   # [a, b)
tabla = clases.value_counts(sort=False).to_frame("pedidos")
tabla["%"] = (tabla["pedidos"] / len(tiempo) * 100).round(1)
acumulado = tabla["pedidos"].cumsum() / len(tiempo) * 100
tabla["% acumulado"] = acumulado.round(1)
tabla

### El histograma muestra la forma

Es la tabla agrupada hecha gráfica: barras pegadas, porque el eje es continuo, y, con intervalos de
igual ancho, la altura es la frecuencia. Mira si la distribución es simétrica, asimétrica o tiene
dos jorobas.

In [ ]:
# [histograma]
# Intervalos de 5 minutos, del 0 al 205: el máximo (200.9) queda dentro
ax = tiempo.plot.hist(bins=range(0, 210, 5), edgecolor="white")
ax.set_title("Tiempo de entrega, en intervalos de 5 minutos")
ax.set_xlabel("Minutos")
ax.set_ylabel("Pedidos")
plt.show()

In [ ]:
# [anchos]
# Los mismos datos con tres anchos de intervalo (hasta 120 min, para ver la forma)
fig, ejes = plt.subplots(1, 3, figsize=(9, 3))
for ax, ancho in zip(ejes, [1, 5, 25]):
    tiempo.plot.hist(bins=range(0, 230, ancho), ax=ax,
                     edgecolor="white", linewidth=0.5)
    ax.set_xlim(0, 120)
    ax.set_title(f"Intervalos de {ancho} min")
    ax.set_xlabel("Minutos")
    ax.set_ylabel("Pedidos")
plt.show()

In [ ]:
# [formas]
# Tres variables de Tingua, tres formas: temperatura en Medellín, temperatura en
# las cinco ciudades (intervalos de 1 °C) y valor del pedido
# .loc[filas, columna]: solo las filas de Medellín, solo la temperatura
medellin = pedidos.loc[pedidos["ciudad"] == "Medellín", "temperatura_c"]
miles = pedidos["valor_pedido"] / 1000
fig, ejes = plt.subplots(1, 3, figsize=(9, 3))
medellin.plot.hist(bins=range(4, 38), ax=ejes[0], edgecolor="white")
ejes[0].set_title("Medellín (°C)")
pedidos["temperatura_c"].plot.hist(bins=range(4, 38), ax=ejes[1],
                                   edgecolor="white")
ejes[1].set_title("Cinco ciudades (°C)")
miles.plot.hist(bins=range(0, 1780, 20), ax=ejes[2], edgecolor="white")
ejes[2].set_xlim(0, 400)   # 16 pedidos pasan de 400 mil: la cola sigue
ejes[2].set_title("Valor (miles de COP)")
for ax in ejes:
    ax.set_ylabel("Pedidos")
plt.show()

## 4 · Tablas cruzadas: dos variables a la vez

Una **tabla cruzada** (o de contingencia) cuenta los pedidos de cada combinación de dos variables
cualitativas. Sus porcentajes se pueden sacar sobre tres totales distintos: el de cada fila, el de
cada columna o el de toda la tabla. Antes de leer un porcentaje, pregunta: **¿sobre qué total?**

In [ ]:
# [cruzada]
pd.crosstab(pedidos["ciudad"], pedidos["lluvia"],
            margins=True, margins_name="Total")

In [ ]:
# [por_fila]
# normalize="index": cada fila (cada ciudad) suma 100 %
por_ciudad = pd.crosstab(pedidos["ciudad"], pedidos["lluvia"],
                         normalize="index")
(por_ciudad * 100).round(1)

In [ ]:
# [por_columna]
# normalize="columns": cada columna suma 100 %
por_lluvia = pd.crosstab(pedidos["ciudad"], pedidos["lluvia"],
                         normalize="columns")
(por_lluvia * 100).round(1)

In [ ]:
# [barras_lluvia]
ax = (por_ciudad["Sí"] * 100).sort_values().plot.barh(width=0.6)
ax.bar_label(ax.containers[0], fmt="{:.1f} %", padding=3)
ax.set_title("Pedidos entregados con lluvia, por ciudad")
ax.set_xlabel("% de los pedidos de cada ciudad")
ax.set_ylabel("")
plt.show()

---
# Práctica guiada

Cuatro ejercicios. Cada celda termina con un verificador que dice qué está bien y, en lo demás, te
da una pista sin darte la respuesta. Reemplaza cada `...` y ejecuta; puedes repetir la celda las
veces que quieras.

## Ejercicio 1 · La tabla de una nominal

Calcula las frecuencias de `medio_pago`: `conteo`, la absoluta, y `porcentaje`, la relativa en %.
Pista: la celda [nominal].

In [ ]:
# [ej1]
conteo = ...       # frecuencia absoluta de medio_pago
porcentaje = ...   # frecuencia relativa de medio_pago, en %
revisar_nominal(conteo, porcentaje)

## Ejercicio 2 · Una ordinal, acumulada

Calcula las frecuencias de `estrato`: `conteo`, en el orden de la escala (del 1 al 6), y
`acumulado`, el % acumulado. Pista: la celda [ordinal].

Después responde abajo, en «Tus respuestas»: ¿qué porcentaje de los pedidos va a direcciones de
estrato 3 o menos? ¿Y de estrato 4 o más?

In [ ]:
# [ej2]
conteo = ...       # pedidos por estrato, en el orden de la escala
acumulado = ...    # % acumulado, del estrato 1 al 6
revisar_acumulada(conteo, acumulado)

## Ejercicio 3 · Una continua, en intervalos

Agrupa `distancia_km` en intervalos de 2 km, cerrados a la izquierda (`[0, 2)`, `[2, 4)`…), que
cubran **todos** los pedidos. Escribe solo la lista de `cortes`: el verificador arma la tabla con
`pd.cut(..., right=False)`, como en la celda [intervalos].

In [ ]:
# [ej3]
cortes = ...       # una lista: [0, 2, 4, ...]
revisar_intervalos(cortes)

Ahora mira las distancias largas con intervalos de medio kilómetro. ¿Qué pasa en 14 km? Pista: el
diccionario de datos.

In [ ]:
# [ej3_grafica]
# .loc[filas, columna]: solo los pedidos de 8 km o más, solo la distancia
largas = pedidos.loc[pedidos["distancia_km"] >= 8, "distancia_km"]
cortes = [8, 8.5, 9, 9.5, 10, 10.5, 11, 11.5, 12, 12.5, 13, 13.5, 14, 14.5, 15]
ax = largas.plot.hist(bins=cortes, edgecolor="white")
ax.set_title("Pedidos de 8 km o más, en intervalos de 0,5 km")
ax.set_xlabel("Kilómetros")
ax.set_ylabel("Pedidos")
plt.show()

## Ejercicio 4 · Qué se pide en cada franja

Cruza `franja` (filas) con `categoria` (columnas) para ver la mezcla de categorías de cada franja:
cada **fila** debe sumar 100 %. Pista: la celda [por_fila].

Después responde abajo: ¿en qué franja pesa más Mercado? ¿Y Restaurante? ¿Por qué crees que pasa?

In [ ]:
# [ej4]
mezcla = ...       # pd.crosstab(filas, columnas, normalize=...)
revisar_cruzada(mezcla)

### Tus respuestas

*Doble clic para escribir aquí.*

1. Ejercicio 2: estrato 3 o menos, … %; estrato 4 o más, … %.
2. Ejercicio 3: en 14 km pasa que…
3. Ejercicio 4: Mercado pesa más en…; Restaurante, en…, porque…